# v040_idf_freq_features — v001 + IDF, pool-frequency, IDF-context and extra address features

| Field | Value |
|---|---|
| **Version** | `v040_idf_freq_features` |
| **Plan group** | C2 (IDF similarities), with C3/C4 (address and number extras) and C5 (pool frequencies, group context) additions |
| **Parent version** | `v001` (`v001_base_model`) |
| **Author** | M3 |
| **Date** | 2026-09-26 |
| **Status** | kept |
| **Hypothesis** | The `idf`, `frequency`, `ctx_idf` and `address_extra` groups separate same-name decoys and dropped-component true pairs, raising val macro F0.5 by ≥ +0.002 over v001 with no slice falling by more than 0.01. |

The v001 pipeline, unchanged (normalisation, blocking, LightGBM parameters, rule grid, seeds),
with four feature groups appended to its 47 features. Only the matcher's inputs change: the
blocking configuration is v001's, so are the candidates (reused from the cache where it exists),
and every difference in the scores comes from the features:

```
raw TSV -> normalize -> blocking                                        (as v001)
        -> features: v001's 47 + idf, frequency, ctx_idf, address_extra  <- this version
        -> LightGBM -> rule retuned on the tune split -> val scored once (as v001)
```

Every version-specific value sits in the parameter cell of §2: a follow-up version (v041, …)
copies this notebook, edits that cell, and rewrites this table, §1 and §8. House rules: every
code cell is preceded by a markdown cell saying what it does and why; the only decision metric
is macro F0.5 on the fixed validation fold.

## 1. Hypothesis

* **Change vs parent:** four feature groups after v001's 47 features (§4.3): `idf`
  (IDF-weighted name and address agreement for every pair), `frequency` (how many pool records
  of the country carry this name / this address), `ctx_idf` (rank and gap of the IDF cosines
  inside the S1 candidate group, number of exact-name candidates) and `address_extra` (reverse
  address containment, number containment, postcode prefix, S1-address-empty flag, address
  length ratio). Nothing else changes.
* **Why it should raise macro F0.5:** v001 loses its F0.5 on two patterns its features cannot
  express (samples in its §6).
  *Same-name decoys*: 48 % of S1 core names are shared and v001's name similarities are 1.0 for
  every record carrying the name; half of its sampled false merges pair an S1 record with a
  name-only pool record (empty address) of another business (`Classic Suisse LLC`,
  `Allied Interests`, `Art Finance Co`, `Synergia & Brothers`). Whether such a name match
  identifies the business depends on how many pool records carry the name (`frequency`, the
  exact-name count of `ctx_idf`).
  *Dropped-component true pairs*: pool records that keep only part of the name or the address
  (`Best Beverage Holdings` → `Best Beverage`, `Cardiology Tri-State Care LLC` →
  `Cardiology Tri-State`, a bare city for an address) lose every missing token at full weight
  in Jaccard and token-set scores. IDF weights make a dropped `holdings` cheap and a dropped
  rare word expensive; reverse containment says "subset" rather than "disagreement".
* **Prediction (falsifiable):** val macro F0.5 ≥ parent + 0.002 (v001: 0.9844), no slice of
  the standard slice report down by more than 0.01, harder-val F0.5 not below the parent's;
  the gain concentrated in the `ambiguous = yes` and `addr_empty = yes` slices (v001: 0.9757
  and 0.9583, against 0.9844 overall); at least one new feature in the gain top 20.
* **Expected cost:** candidates, candidate recall and the F0.5 ceiling identical to v001;
  feature and scoring time up (per-country pool statistics, IDF token matrices), val
  `score_seconds` below 1.5× the parent's; peak RSS below the 13 GB guard.
* **Risk:** pool statistics depend on the pool they are counted on. Training pairs are featured
  against the whole fit pool (~6.2M records), tune and val against ~2.1M each, test against the
  per-country test pools (5.8 pool records per S1 against 4.7 in train). Raw frequency counts
  shift with pool size, which neither val nor harder-val (same pool) exposes; §6.2 shows the
  class means of every new feature on val, §8 asks for the verdict.
* **Discard if:** the 13 §3 rule, applied automatically in §7, says DROP (F0.5 not above the
  parent, or a slice down by more than 0.01).

## 2. Setup

All imports first. The cell after holds the parameters of this version; the one after that
derives the folders, reads the parent's `metrics.json` (the reference of every comparison,
never retyped) and checks that this version is exactly "parent + `NEW_GROUPS`".

In [1]:
import inspect
import json
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd
from IPython.display import display

from entity_resolution import config as C
from entity_resolution.blocking import PASS_BITS
from entity_resolution.evaluate import (
    entity_counts,
    entity_f05_from_counts,
    error_samples,
    harder_fold,
    pair_in,
    positions,
    slice_report,
)
from entity_resolution.features import (
    DEFAULT_GROUPS,
    FEATURE_COLUMNS,
    REGISTRY,
    build_features,
    feature_names,
)
from entity_resolution.pipeline import (
    PipelineConfig,
    fit,
    load_normalised,
    peak_rss_gb,
    run_fold,
    run_test,
)
from entity_resolution.split import hash_unit, load_fold
from entity_resolution.tracking import log_result, timed

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 30)
t_start = time.time()  # notebook wall time, printed at the end

### Parameters of this version (the only code cell a follow-up version edits)

`FEATURE_GROUPS` is the parent's feature set (`DEFAULT_GROUPS`: v001's 8 groups, 47 features)
followed by `NEW_GROUPS`. `cfg` keeps every other `PipelineConfig` default, which is v001's
configuration. `RUN_TEST` stays `False` unless M1 shortlists this version for an upload (§9).

In [2]:
VERSION = "v040_idf_freq_features"  # this folder under experiments/
PARENT = "v001_base_model"          # logged version compared against (its metrics.json)
GROUP = "C2"                        # plan ID: C2 IDF similarities, with C3-C5 additions
OWNER = "M3"
NEW_GROUPS = ("idf", "frequency", "ctx_idf", "address_extra")  # feature groups added here
# the parent's groups first, then the new ones; the filter keeps the tuple free of repeats
# should DEFAULT_GROUPS ever list a new group itself (feature_names rejects repeated groups)
FEATURE_GROUPS = tuple(g for g in DEFAULT_GROUPS if g not in NEW_GROUPS) + NEW_GROUPS
CHANGE = "v001 + idf, frequency, ctx_idf, address_extra feature groups"
HYPOTHESIS = ("the idf, frequency, ctx_idf and address_extra groups separate same-name decoys "
              "and dropped-component true pairs, raising val macro F0.5 by >= +0.002 over v001 "
              "with no slice falling by more than 0.01")
RUN_TEST = False  # test inference only for shortlisted versions (§9)

cfg = PipelineConfig(feature_groups=FEATURE_GROUPS)  # every other field: v001's defaults

Derived paths and the parent's record. `feature_names` raises at once on an unknown or repeated
group, so a group that is not implemented yet fails here rather than after minutes of blocking.
The groups other than `NEW_GROUPS` must equal the parent's logged `feature_groups`, or the
comparisons of §5 would not isolate the new groups. The printed configuration is saved with the
model (`artifacts/config.json`) and logged in `metrics.json`.

In [3]:
EXP_DIR = C.EXPERIMENTS / VERSION
ARTIFACTS = EXP_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
PARENT_DIR = C.EXPERIMENTS / PARENT
PARENT_ARTIFACTS = PARENT_DIR / "artifacts"  # gitignored: only on the machine that ran it
parent_record = json.loads((PARENT_DIR / "metrics.json").read_text(encoding="utf-8"))
parent_metrics = parent_record["metrics"]  # the parent's 13 §2.2 record
timings: dict[str, float] = {}  # the eight 13 §2.2 stage times of this run

names = feature_names(cfg.feature_groups)  # raises now on an unknown or repeated group
new_features = [c for g in NEW_GROUPS for c in FEATURE_COLUMNS[g]]
base_groups = [g for g in cfg.feature_groups if g not in NEW_GROUPS]
if base_groups != parent_metrics["feature_groups"]:
    raise ValueError(f"groups other than NEW_GROUPS {base_groups} differ from the parent's "
                     f"{parent_metrics['feature_groups']}: fix FEATURE_GROUPS (parameters)")
print(f"{VERSION}: parent {parent_record['version']} (local F0.5 "
      f"{parent_metrics['f_beta']:.4f}); {len(names)} features = "
      f"{len(names) - len(new_features)} parent + {len(new_features)} new")
print(json.dumps(cfg.record(), indent=1)[:3000])

v040_idf_freq_features: parent v001 (local F0.5 0.9844); 70 features = 47 parent + 23 new
{
 "normalise": {
  "transliterate": true,
  "strip_legal": true,
  "expand_abbrev": true,
  "region_map": null,
  "chunk_rows": 1000000,
  "learn_token_map": true,
  "token_map_min_count": 3,
  "token_map_min_share": 0.5
 },
 "blocking": {
  "exact_keys": [
   "name_core",
   "name_sorted",
   "name_squash"
  ],
  "exact_max_group": 50,
  "name_char": {
   "column": "name_core",
   "analyzer": "char_wb",
   "ngram": [
    3,
    3
   ],
   "top_k": 10,
   "min_sim": 0.5,
   "max_df": 0.2,
   "min_df": 2,
   "sublinear_tf": true,
   "pool_max_addr_tokens": 3,
   "max_df_abs": 20000
  },
  "name_addr_word": {
   "column": "name_addr",
   "analyzer": "word",
   "ngram": [
    1,
    2
   ],
   "top_k": 25,
   "min_sim": 0.2,
   "max_df": 0.01,
   "min_df": 2,
   "sublinear_tf": true,
   "pool_max_addr_tokens": null,
   "max_df_abs": 10000
  },
  "addr_char": null,
  "max_per_s1": 60,
  "s1_chunk": 5

## 3. Data

The fixed validation split (`split.load_fold`, 20 % of Source 1 by id hash, seed 42), as in
v001: every version scores the same held-out entities against the same pool, so local F0.5
values compare. Everything trainable is fitted on the `train` fold only (inner split: fit side
for LightGBM, tune side for early stopping and the rule). The val pool keeps the records matched
to *other* val entities: they are the decoys that make singletons and same-name businesses hard,
and part of the pool the new IDF and frequency statistics are counted on.

In [4]:
with timed("load", timings):
    train = load_fold("train", columns=[])  # ids only: the pipeline reads normalised records
    val = load_fold("val")                  # raw columns kept for the error samples
pd.DataFrame([train.summary(), val.summary()])

,fold,s1,s2,s3,true_pairs,singleton_share
0,train,1765488,4026279,4229875,6110753,0.0558
1,val,441333,1008337,1055728,1527612,0.0559


## 4. Method

### 4.1 Normalisation (as in parent)

v001's static rules and learned transliteration token map (`normalize.py`, v001 §4.1) with the
same `NormaliseConfig`: the normalised records come from the same Parquet cache, and the token
map, learned from the train fold's true pairs, is the same map.

### 4.2 Blocking (as in parent)

Same `BlockingConfig` (v001 §4.2): exact `name_core` / `name_sorted` / `name_squash` keys, name
char 3-grams against short-address pool records, and the name + address word TF-IDF pass,
capped at 60 candidates per S1, per country. Candidates are cached by tag, ids and
blocking-config hash, so this version reuses v001's candidate sets where that cache exists, and
its candidate recall and F0.5 ceiling must equal v001's (§5.5 checks). The cell compares this
version's blocking and model configuration with the parent's logged ones: both lines should read
"identical to the parent".

In [5]:
def config_diff(ours: dict, theirs: dict) -> dict:
    """Keys whose values differ between two JSON-ready configurations: {key: (ours, theirs)}."""
    return {k: (ours.get(k), theirs.get(k)) for k in sorted(set(ours) | set(theirs))
            if ours.get(k) != theirs.get(k)}


cfg_json = cfg.record()  # JSON round trip: tuples become lists, as in the parent's record
for stage, ours, theirs in (("blocking", cfg_json["blocking"], parent_metrics["blocking_config"]),
                            ("model", cfg_json["model"], parent_metrics["model_params"])):
    diff = config_diff(ours, theirs)
    print(f"{stage}: " + ("identical to the parent" if not diff
                          else f"DIFFERS from the parent (ours, parent): {diff}"))

blocking: identical to the parent
model: identical to the parent


### 4.3 Pair features (`features.py`): v001's 47 + four groups

v001's groups are unchanged (blocking evidence, fuzzy names, name tokens, legal form, numbers,
address, group context, meta; its §4.3). What the new groups measure and why each should help
follows; names in brackets are those of the implementation at the time of writing, and the cell
below prints the authoritative column lists and definitions from the library.

*Pool statistics.* `idf`, `frequency` and `ctx_idf` read counts over the pool records of the
pair's country (`features.pool_stats`), computed once per pool and handed to every chunk, so
they never depend on the chunking. Only pool records are counted: the pool is complete in every
setting, while Source 1 is sampled on the fit side.

**`idf`**: IDF-weighted token agreement for **every** pair (the blocking cosines exist only for
the pairs their pass proposed). A token weighs `ln((1 + N) / (1 + df)) + 1`, `df` being the
number of the country's `N` pool records that hold it; NaN when either string is empty.
* *name and address IDF cosines* (`idf_name_cos`, `idf_addr_cos`): a shared rare word
  (`faclara`, a street name, a house number) counts far more than a shared `enterprises`,
  `road`, city or state token. Two businesses of the same city stop looking alike through their
  addresses; two records of one business that share their rare words look alike even when the
  generic words differ.
* *rarest shared token* (`idf_*_top`): the largest shared IDF over the largest possible one. One
  shared very rare token is strong evidence on its own; sharing only common tokens is not.
* *coverage per side* (`idf_*_cover_l`, `idf_*_cover_r`): the share of each side's IDF mass
  found on the other side. A pool record that dropped components keeps a full coverage of its
  own mass (everything it says is in the S1 record) and the S1 side loses only the weight of what
  was dropped (`Best Beverage Holdings` → `Best Beverage` loses a cheap `holdings`). A decoy
  that shares only generic tokens has a low coverage on both sides.

**`frequency`**: how many pool records of the country hold exactly this name or this address
(`freq_name_l`, `freq_name_r`, `freq_addr_l`, `freq_addr_r`: the S1 record's string and the pool
record's, itself included; NaN when that side is empty). A name held by 30 pool records is weak
evidence on its own, most of them being other businesses; a name held by one or two is strong
even when the address is missing, the case of v001's name-only false merges and misses. On the
address side a high count marks shared buildings, malls and registered-agent addresses, where an
address match does not identify the business. v001 sees the competition inside the S1 group
(`ctx_n_cands`, ranks) but not how common a name is in the pool.

**`ctx_idf`**: the S1 group's view of the IDF cosines, which the decision layer also sees
(pool-side 1-to-1, `tau_rel`).
* *rank and gap to the group's best* of the name and address IDF cosines (`ctx_rank_idf_*`,
  `ctx_gap_idf_*`): v001 ranks on `sim_name_char`, NaN for every pair the char pass did not
  propose, and on `ad_token_set`, which gives 1.0 to every pool address whose tokens are a
  subset of the S1 address's (a bare city included); these ranks see every candidate and weigh
  its tokens by rarity.
* *exact-name candidates of the group* (`ctx_n_same_name`): how many same-name decoys this S1
  record faces. When there are many, only the address can decide.

**`address_extra`**: what v001's address and numeric groups say only indirectly, or not at all.
* *reverse containment* (`ad_contain_r`): the share of the pool address tokens found in the S1
  address (v001's `ad_contain` goes S1 → pool only). A pool address reduced to a city
  (`Agra, UP`) is fully contained: dropped components, not disagreement; components only the
  pool has (a suite, a landmark) show up as a lower share.
* *number containment* (`num_contain_l`, `num_contain_r`): the share of each side's address
  numbers found among the other's. A number one side dropped keeps the pair plausible, a changed
  number does not: house and unit numbers separate same-street decoys (`16-17 163 Street` vs
  `16-19 163 ST`, `8 Web Road` vs `29 WEB ROAD` among v001's false merges).
* *postcode prefix* (`postcode_prefix_eq`): equal first three postcode characters, the same
  postal area when a typo in the last digits or a neighbouring code breaks `postcode_eq`.
* *S1 address empty* (`addr_empty_l`): v001 flags only an empty pool address (`addr_empty_r`);
  with an empty S1 address every address similarity is NaN and the model cannot tell which side
  is empty.
* *address length ratio* (`addr_len_ratio`): fewer over more distinct address tokens. A high
  similarity between a bare city and a full street address is weaker evidence than one between
  two full addresses.

The groups of this version with their column counts (new groups flagged), then the full column
list and the docstring of each new group, read from `FEATURE_COLUMNS` and `REGISTRY`: the column
names are never typed in this notebook.

In [6]:
groups_table = pd.DataFrame(
    [(g, g in NEW_GROUPS, len(FEATURE_COLUMNS[g]), ", ".join(FEATURE_COLUMNS[g]))
     for g in cfg.feature_groups], columns=["group", "new", "n", "features"])
display(groups_table)
for g in NEW_GROUPS:  # the table truncates long lists: print each new group in full
    print(f"=== {g} ({len(FEATURE_COLUMNS[g])} columns): {', '.join(FEATURE_COLUMNS[g])}")
    print(inspect.getdoc(REGISTRY[g]), end="\n\n")

,group,new,n,features
0,blocking,False,6,"pass_exact, pass_name_char, pass_name_addr, sim_name_cha..."
1,name_fuzzy,False,10,"nm_ratio, nm_partial, nm_token_sort, nm_token_set, nm_jw..."
2,name_tokens,False,8,"tok_jaccard, tok_dice, tok_common, tok_len_l, tok_len_r,..."
3,legal,False,3,"legal_eq, legal_missing_l, legal_missing_r"
4,numeric,False,4,"num_jaccard, num_shared_any, num_first_eq, postcode_eq"
5,address,False,8,"ad_token_set, ad_partial, ad_ratio, ad_jaccard, ad_conta..."
6,context,False,5,"ctx_rank_name, ctx_gap_name, ctx_rank_addr, ctx_gap_addr..."
7,meta,False,3,"is_s3, non_latin_r, len_ratio_name"
8,idf,True,8,"idf_name_cos, idf_name_top, idf_name_cover_l, idf_name_c..."
9,frequency,True,4,"freq_name_l, freq_name_r, freq_addr_l, freq_addr_r"


=== idf (8 columns): idf_name_cos, idf_name_top, idf_name_cover_l, idf_name_cover_r, idf_addr_cos, idf_addr_top, idf_addr_cover_l, idf_addr_cover_r
IDF-weighted token agreement for every pair (C2, TRACKER #16).

The blocking cosines (sim_*) exist only for the pairs their pass proposed; these exist
for every pair. A token weighs idf = ln((1 + N) / (1 + df)) + 1, df counted over the pool
records of the pair's country (``pool_stats``), so a shared rare token ("zenith", a
house number) counts far more than a shared "cafe" or "road".

idf_name_cos                        cosine of the name_core idf vectors
idf_name_top                        largest shared idf / largest possible idf; 0 if none
idf_name_cover_l, idf_name_cover_r  share of that side's idf mass found on the other
idf_addr_*                          the same on addr_norm
All are NaN when either string is empty.

=== frequency (4 columns): freq_name_l, freq_name_r, freq_addr_l, freq_addr_r
How many pool records of the country sha

### 4.4 Matching model (as in parent)

LightGBM (MIT) with v001's parameters (§4.2 prints them identical), trained on the same 200k
fit-side S1 entities with all their candidates against the whole fit pool, early-stopped on the
same 50k tune-side sample, no class reweighting. Only the column contract grows: the model
stores `feature_names(FEATURE_GROUPS)` and refuses any other frame.

### 4.5 Decision rule (as in parent, retuned on the tune split)

Same rule family and grid (pool-side 1-to-1, `tau_abs`, `tau_rel`, `tau_single`,
`max_matches`), grid-searched again for macro F0.5 on all tune-side S1 entities: the new
features change the probabilities, so the parent's thresholds are not reused.

## 5. Evaluation on the validation fold

### 5.1 Fit on the train fold

`fit` runs the whole training side: normalisation cache, token map, blocking of the fit, stop and
tune sides (cached), features with the new groups (their pool statistics counted on each side's
pool), LightGBM, scoring of the tune side and the rule grid. The val fold is not touched.
`features_seconds` (fit- and stop-side features, stop-side blocking included) joins the logged
timings; v001 did not log it, so val `score_seconds` (§5.2) is the like-for-like cost figure.

In [7]:
fit_timings: dict[str, float] = {}
t0 = time.time()
fitted = fit(cfg, train, ARTIFACTS, fit_timings)
for k in ("features_seconds", "fit_seconds", "tune_seconds"):
    timings[k] = fit_timings.get(k, 0.0)
print(f"fit() total {time.time() - t0:.0f} s; stages: {fit_timings}")
print("token map:", fitted.info["token_map_size"], "tokens; parent:",
      parent_metrics.get("token_map_size"))
print("training:", fitted.info["fit_info"])
print("rule:", fitted.rule, "| parent:", parent_metrics["rule"])

fit() total 1328 s; stages: {'normalise_seconds': 6.58, 'fit_load_seconds': 16.92, 'fit_blocking_seconds': 9.83, 'stop_load_seconds': 7.9, 'stop_blocking_seconds': 2.4, 'features_seconds': 182.63, 'fit_seconds': 496.36, 'tune_load_seconds': 7.62, 'tune_blocking_seconds': 6.13, 'score_seconds': 536.34, 'tune_seconds': 25.99}
token map: 536 tokens; parent: 536
training: {'rows': 6719931, 'positive_rate': 0.09974239318826339, 'best_iteration': 1999, 'tune_logloss': 0.00839177345205442, 'tune_auc': 0.9998685846617594, 'fit_seconds': 496.36}
rule: DecisionRule(tau_abs=0.445, tau_rel=0.0, tau_single=0.495, max_matches=11, one_to_one=True) | parent: {'tau_abs': 0.47, 'tau_rel': 0.0, 'tau_single': 0.52, 'max_matches': 11, 'one_to_one': True}


Blocking quality of the three training sides (the candidates are v001's, so these rows must
match its fit / stop / tune rows), the 20 most important features (gain share, new ones
flagged), the rank and gain of **every** new feature, and the best rules of the tune grid. A new
feature with zero gain only adds cost (07 §6 drops features at zero gain for three versions).
The full importance table goes to `artifacts/importance.csv`.

In [8]:
blk = pd.DataFrame({side: fitted.info[f"{side}_blocking"] for side in ("fit", "stop", "tune")}).T
display(blk[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean",
             "candidates_p95"]])
group_of = {c: g for g in cfg.feature_groups for c in FEATURE_COLUMNS[g]}
importance = fitted.matcher.importance().rename("gain").to_frame()
importance["rank"] = np.arange(1, len(importance) + 1)  # 1 = largest gain
importance["group"] = importance.index.map(group_of)
importance["new"] = importance["group"].isin(NEW_GROUPS)
importance.to_csv(ARTIFACTS / "importance.csv")
display(importance.head(20))
new_importance = importance[importance["new"]]
print(f"new features: {len(new_importance)}; gain share {new_importance['gain'].sum():.4f}; "
      f"in the top 20: {int((new_importance['rank'] <= 20).sum())}; "
      f"zero gain: {int((new_importance['gain'] == 0).sum())}")
display(new_importance)
fitted.tune_table.sort_values("f_beta", ascending=False).head(10)

,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
fit,0.973580,0.997445,0.990903,33.764425,42.0
stop,0.990733,0.999216,0.997052,32.803258,60.0
tune,0.990317,0.999275,0.996959,32.963458,60.0


,gain,rank,group,new
feature,,,,
ad_token_set,0.379806,1,address,False
sim_name_addr_word,0.113734,2,blocking,False
core_token_set,0.058406,3,name_fuzzy,False
ctx_gap_addr,0.053899,4,context,False
ctx_rank_addr,0.038953,5,context,False
core_jw,0.033265,6,name_fuzzy,False
num_contain_l,0.026904,7,address_extra,True
idf_addr_cos,0.024161,8,idf,True
squash_ratio,0.023343,9,name_fuzzy,False


new features: 23; gain share 0.1557; in the top 20: 6; zero gain: 2


,gain,rank,group,new
feature,,,,
num_contain_l,0.026904,7,address_extra,True
idf_addr_cos,0.024161,8,idf,True
ad_contain_r,0.017109,12,address_extra,True
ctx_rank_idf_addr,0.014731,14,ctx_idf,True
idf_addr_cover_r,0.012993,15,idf,True
num_contain_r,0.011651,17,address_extra,True
idf_name_cover_r,0.007905,21,idf,True
freq_name_r,0.005828,23,frequency,True
idf_name_top,0.005713,24,idf,True


,tau_abs,tau_rel,tau_single,max_matches,one_to_one,f_beta,n_pred,pair_precision,pair_recall,match_rate,stage
632,0.400,0.0,0.400,11,True,0.987172,1487220,0.996025,0.970158,0.943212,grid
506,0.380,0.0,0.380,11,True,0.987163,1488365,0.995812,0.970697,0.943287,grid
509,0.380,0.0,0.430,11,True,0.987159,1488275,0.995841,0.970667,0.943083,grid
635,0.400,0.0,0.450,11,True,0.987159,1487119,0.996056,0.970123,0.942986,grid
758,0.420,0.0,0.420,11,True,0.987154,1486140,0.996197,0.969621,0.943133,grid
383,0.360,0.0,0.410,11,True,0.987150,1489400,0.995618,0.971183,0.943169,grid
638,0.400,0.0,0.500,11,True,0.987137,1487021,0.996086,0.970088,0.942768,grid
3906,0.410,0.0,0.460,11,True,0.987135,1486544,0.996146,0.969835,0.942940,refine
3907,0.415,0.0,0.465,11,True,0.987127,1486289,0.996183,0.969705,0.942918,refine
512,0.380,0.0,0.480,11,True,0.987124,1488172,0.995869,0.970627,0.942859,grid


### 5.2 Validation fold

`run_fold` blocks, scores and decides the val fold once with the frozen rule. Primary metric:
**macro F0.5 over all val S1 entities, singletons included** (`evaluate.score_pairs`, equal to
`metrics.breakdown`). Blocking quality: candidate pair recall, entity recall and the F0.5
ceiling of a perfect matcher on these candidates, all three equal to the parent's.

In [9]:
t0 = time.time()
metrics, val_pairs, val_scored, val_matches = run_fold(cfg, fitted, val)
for k in ("normalise_seconds", "blocking_seconds", "score_seconds", "decide_seconds"):
    timings[k] = metrics.get(k, 0.0)
print(f"run_fold {time.time() - t0:.0f} s")
pd.Series(metrics)

run_fold 845 s


f_beta                    0.986978
f_beta_singletons         0.987522
f_beta_matched            0.986946
pair_precision            0.996408
pair_recall               0.968893
entities             441333.000000
singletons            24684.000000
cand_recall               0.990570
entity_recall             0.999330
ceiling_f_beta            0.997067
cands_mean               32.970535
cands_p95                60.000000
normalise_seconds         7.480000
blocking_seconds          4.120000
score_seconds           813.810000
decide_seconds           10.590000
dtype: float64

The scored candidate pairs, the val matches and the candidate pairs (ids and pass bits only) go
to `artifacts/` (gitignored): a child version compares its slices and transitions against this
one from `val_matches.parquet` without re-running it, as §5.6 and §6.3 do with the parent's.

In [10]:
val_scored.to_parquet(ARTIFACTS / "scored_val.parquet", index=False)
val_matches.to_parquet(ARTIFACTS / "val_matches.parquet", index=False)
val_pairs[[C.S1_ID, C.ENTITY_ID, "pass"]].to_parquet(ARTIFACTS / "val_pairs.parquet",
                                                    index=False)
print({p.name: f"{p.stat().st_size / 1e6:.1f} MB" for p in sorted(ARTIFACTS.glob("*.parquet"))})

{'scored_val.parquet': '207.7 MB', 'val_matches.parquet': '19.0 MB', 'val_pairs.parquet': '149.2 MB'}


### 5.3 Blocking passes and slices

Recall of each blocking pass on the val fold (a pair can come from several passes) and the
standard slice report (11 §7): country, source, Indic-script pool names, domain forms, ambiguous
core names, singletons, number of true matches, postcode, empty addresses. The normalised val
records loaded here also feed §5.6 and §6.

In [11]:
is_true = pair_in(val_pairs, val.pairs)  # candidate pair is a true pair (reused in §6.2)
pass_rows = []
for name, bit in PASS_BITS.items():
    in_pass = (val_pairs["pass"].to_numpy() & bit) != 0
    if in_pass.any():
        pass_rows.append((name, in_pass.sum() / len(val.s1),
                          (in_pass & is_true).sum() / len(val.pairs)))
display(pd.DataFrame(pass_rows, columns=["pass", "pairs_per_s1", "recall"]))
s1n_val = load_normalised("train", (1,), cfg, val.s1[C.ENTITY_ID], fitted.token_map)
pooln_val = load_normalised("train", (2, 3), cfg, pd.concat([val.s2, val.s3])[C.ENTITY_ID],
                            fitted.token_map)
slices = slice_report(val_matches, val, s1n_val, pooln_val)
slices.to_csv(ARTIFACTS / "slices.csv", index=False)
display(slices)

,pass,pairs_per_s1,recall
0,exact_core,7.244736,0.564582
1,exact_sorted,7.333775,0.595844
2,exact_squash,7.311683,0.594464
3,name_char,7.387163,0.045524
4,name_addr_word,21.615297,0.982539


,family,slice,entities,f_beta,pair_precision,pair_recall,n_true,n_pred,tp
0,country,India,176522,0.986510,0.996542,0.966235,611167,592580,590531
1,country,US,264811,0.987291,0.996320,0.970665,916445,892847,889561
2,country,all,441333,0.986978,0.996408,0.968893,1527612,1485427,1480092
3,source,S2,384042,0.980052,0.996527,0.971607,739443,720952,718448
4,source,S3,388292,0.976994,0.996297,0.966346,788169,764475,761644
5,non_latin,yes,54312,0.986113,0.998429,0.956165,204745,196078,195770
6,non_latin,no,387021,0.987100,0.996101,0.970863,1322867,1289349,1284322
7,non_latin,all,441333,0.986978,0.996408,0.968893,1527612,1485427,1480092
8,domain_form,yes,82591,0.988757,0.996739,0.969438,349356,339787,338679
9,domain_form,no,358742,0.986569,0.996310,0.968731,1178256,1145640,1141413


### 5.4 Harder validation

Test has 5.8 pool records per S1 against 4.7 in train (11 §6), so the same frozen rule is also
scored on a val variant that drops 20 % of the S1 entities but keeps their pool records (they
become unowned decoys). A version whose harder score falls while val rises is buying recall with
false merges. The pool is unchanged here, so the new pool statistics are the same as on val:
this measures robustness to extra decoys, not to pool size.

In [12]:
harder_metrics = run_fold(cfg, fitted, harder_fold(val), tag="harder")[0]  # frames dropped
print({k: round(v, 4) for k, v in harder_metrics.items() if k.startswith(("f_beta", "pair"))})

{'f_beta': 0.9864, 'f_beta_singletons': 0.9841, 'f_beta_matched': 0.9866, 'pair_precision': 0.9956, 'pair_recall': 0.9692}


### 5.5 Comparison with the parent

The parent's numbers are read from its `metrics.json`. `cand_recall` is a sanity row: the
candidates are the parent's, so it must not move. The second table puts stage times and peak RSS
next to the parent's (13 §3 treats a doubled run time or RSS as a reason to INVESTIGATE);
`features_seconds` is NaN for v001, which did not log it, and `blocking_seconds` is small when
the candidates come from the cache.

In [13]:
COMPARE_KEYS = ("f_beta", "f_beta_singletons", "f_beta_matched", "pair_precision",
                "pair_recall", "harder_f_beta", "tune_f_beta", "cand_recall")
COST_KEYS = ("features_seconds", "fit_seconds", "blocking_seconds", "score_seconds",
             "decide_seconds", "peak_rss_gb")
this_numbers = {**metrics, **timings, "harder_f_beta": harder_metrics["f_beta"],
                "tune_f_beta": float(fitted.tune_table["f_beta"].max()),
                "peak_rss_gb": peak_rss_gb()}


def versus_parent(keys: tuple[str, ...]) -> pd.DataFrame:
    """The parent's value (NaN when its record lacks the key) and this version's, per key."""
    return pd.DataFrame({"parent": [float(parent_metrics.get(k, np.nan)) for k in keys],
                         "this": [float(this_numbers.get(k, np.nan)) for k in keys]},
                        index=list(keys))


comparison = versus_parent(COMPARE_KEYS).assign(delta=lambda d: d["this"] - d["parent"])
display(comparison.round(5))
versus_parent(COST_KEYS).assign(ratio=lambda d: d["this"] / d["parent"]).round(2)

,parent,this,delta
f_beta,0.98436,0.98698,0.00261
f_beta_singletons,0.98400,0.98752,0.00352
f_beta_matched,0.98439,0.98695,0.00256
pair_precision,0.99521,0.99641,0.00120
pair_recall,0.96373,0.96889,0.00517
harder_f_beta,0.98383,0.98643,0.00260
tune_f_beta,0.98465,0.98717,0.00253
cand_recall,0.99057,0.99057,0.00000


,parent,this,ratio
features_seconds,NaN,182.63,NaN
fit_seconds,335.79,496.36,1.48
blocking_seconds,448.19,4.12,0.01
score_seconds,270.24,813.81,3.01
decide_seconds,6.79,10.59,1.56
peak_rss_gb,5.92,5.28,0.89


### 5.6 Slices against the parent

This version's slice report next to the parent's, on the same val fold and the same normalised
records. The parent's slices are recomputed from its `artifacts/val_matches.parquet` when that
file exists (§5.2 saves one for this version's children), else read from its
`artifacts/slices.csv` (v001 wrote one), else unavailable: artifacts are gitignored and exist
only on the machine that ran the parent. `max_slice_drop` is the largest `parent − this` slice
F0.5 (negative when every slice rose); it is NaN when the parent's slices are unavailable, which
the output says and §7 treats as "no drop". The joined table goes to
`artifacts/slices_vs_parent.csv`.

In [14]:
parent_matches_path = PARENT_ARTIFACTS / "val_matches.parquet"
parent_matches = pd.read_parquet(parent_matches_path) if parent_matches_path.exists() else None
if parent_matches is not None:
    parent_slices = slice_report(parent_matches, val, s1n_val, pooln_val)
    source = f"recomputed from {parent_matches_path}"
elif (PARENT_ARTIFACTS / "slices.csv").exists():
    parent_slices = pd.read_csv(PARENT_ARTIFACTS / "slices.csv",
                                dtype={"family": str, "slice": str},
                                keep_default_na=False, na_values=[""])  # "0" stays a slice name
    source = f"read from {PARENT_ARTIFACTS / 'slices.csv'}"
else:
    parent_slices = None
    source = f"UNAVAILABLE (no val_matches.parquet or slices.csv in {PARENT_ARTIFACTS})"

slice_cmp = slices[["family", "slice", "entities", "f_beta", "pair_precision", "pair_recall"]]
if parent_slices is None:
    slice_cmp = slice_cmp.assign(f_beta_parent=np.nan)
else:
    slice_cmp = slice_cmp.merge(
        parent_slices[["family", "slice", "f_beta"]].rename(columns={"f_beta": "f_beta_parent"}),
        on=["family", "slice"], how="left")
slice_cmp["delta"] = slice_cmp["f_beta"] - slice_cmp["f_beta_parent"]
drop = -slice_cmp["delta"]  # > 0: the slice fell
max_slice_drop = float(drop.max())  # NaN when no parent slice is known
slice_cmp.to_csv(ARTIFACTS / "slices_vs_parent.csv", index=False)
print(f"parent slices: {source}")
if np.isnan(max_slice_drop):
    print("max_slice_drop = NaN: the slice condition of the 13 §3 rule is skipped in §7")
else:
    worst = slice_cmp.loc[drop.idxmax()]
    print(f"max_slice_drop = {max_slice_drop:+.4f} ({worst['family']} / {worst['slice']}, "
          f"{worst['entities']} entities)")
display(slice_cmp[["family", "slice", "entities", "f_beta_parent", "f_beta", "delta",
                   "pair_precision", "pair_recall"]])

parent slices: recomputed from C:\Users\user\OneDrive\Documents\projects\business_entity_resolution\experiments\v001_base_model\artifacts\val_matches.parquet
max_slice_drop = -0.0019 (n_matches / 5+, 114843 entities)


,family,slice,entities,f_beta_parent,f_beta,delta,pair_precision,pair_recall
0,country,India,176522,0.983474,0.986510,0.003036,0.996542,0.966235
1,country,US,264811,0.984959,0.987291,0.002332,0.996320,0.970665
2,country,all,441333,0.984365,0.986978,0.002614,0.996408,0.968893
3,source,S2,384042,0.975956,0.980052,0.004097,0.996527,0.971607
4,source,S3,388292,0.973105,0.976994,0.003889,0.996297,0.966346
5,non_latin,yes,54312,0.984014,0.986113,0.002099,0.998429,0.956165
6,non_latin,no,387021,0.984414,0.987100,0.002686,0.996101,0.970863
7,non_latin,all,441333,0.984365,0.986978,0.002614,0.996408,0.968893
8,domain_form,yes,82591,0.986434,0.988757,0.002323,0.996739,0.969438
9,domain_form,no,358742,0.983888,0.986569,0.002681,0.996310,0.968731


## 6. Error analysis

### 6.1 Error kinds

Samples of the four error kinds, both records side by side with the model probability: false
merges on matched entities, missed true pairs, matched entities predicted empty (false
singletons) and predictions on true singletons, each count next to the parent's. The counts say
where the lost F0.5 sits; the samples name the pattern for the next version.

In [15]:
counts: dict[str, int] = {}
parent_errors = parent_metrics.get("errors", {})
for kind in ("false_merge", "missed", "false_singleton", "singleton_merge"):
    sample = error_samples(val_matches, val, kind, n=10, scored=val_scored)
    counts[kind] = len(error_samples(val_matches, val, kind, n=10**9))
    print(f"--- {kind}: {counts[kind]} pairs (parent {parent_errors.get(kind, 'n/a')})")
    display(sample)

--- false_merge: 5021 pairs (parent 6685)


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-104216715,S2-928685796,0.608247,Toss Holdings,"60, 2Nd Main Road, Thirumalai Nagar Annex, Saidapet, Kan...",Toss Systems,"67, 2ND MAIN ROAD, THIRUMALAI NAGAR ANNE, SAIDAPET, Tami..."
1,S1-137695892,S3-697508841,0.915888,Classic Suisse LLC,"Phoenix, 15801 48th Street, AZ, Unit 1216",Classic Suisse LLC,
2,S1-193648498,S2-676218258,0.720746,Chamunda Brothers,"Bangalore North, Karnataka, 12Th Main, 3Rd Phase, Peenya...",Chamunda Brothers Private Limited,"NO.205/B-8A, 12TH MAIN, 3RD PHASE, PEENYA INDUSTRIAL ARE..."
3,S1-507502381,S2-390328928,0.806176,Mridul Traders Private Limited,"No 264, Akshaya Complex, 1St Floore, 8Th Block, Nagarabh...",MRIDUL PRIVATE LIMITED-CENTER,
4,S1-514791271,S3-916567280,0.463590,Trading Competition Vision Limited,"521A Lane No 23Western Avenur Sainik Farm, New Delhi, So...",Trading Competition,
5,S1-557029797,S3-570082564,0.753184,Hein & Ohara Corp,"48 Auburn Road, Peru, ME",Hein & Ohara Corp,
6,S1-584271054,S2-215510386,0.976507,Art Finance Co,"B-1/134 (Pvt. Cabin No.-1), Second Floor Yamuna Vihar, D...",Art Finance,
7,S1-625512706,S3-793416377,0.838940,Kent Future Inc,"3227 67th Terrace, Kansas City, KS",Kent Future,
8,S1-738128729,S2-959360977,0.960032,Faclara Capital Partners LLC,"8 Web Road, Georgetown, MA",Faclara Capital Partners,"29 WEB ROAD, MA, GEORGETOWN"
9,S1-915651212,S2-941036462,0.541439,Shyam Tech Private Limited,"6-20-83, 9/1, Arundelpet, Guntur, Andhra Pradesh",SHYAM TECH PRIVATE LIMITED Enterprises,


--- missed: 46202 pairs (parent 53904)


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-165098328,S3-923481956,0.066554,Kaur Global Digital Inc,"115 Mckinley Avenue, Sapulpa, OK",Vantageyuma Sys,"115 Mckinley Avenue, Sapulpa, OK"
1,S1-172965217,S2-842589544,NaN,Laxmi Blue Producer Private Limited,"Fl. 200, Bl.A/1, Alcon Acacia, Sn. 7/20, 24, 25, Nr. Tal...",Private Laxmi Brfs Producer Limited,"DOOR NO 626 FL. 200, PUNE, Maharashtra"
2,S1-223235489,S3-693515795,0.388515,Cardiology Tri-State Care LLC,"4313 Gerald Road, Ashtabula, OH",Cardiology Tri-State,
3,S1-242332414,S2-751909198,0.360368,Jay It Private Limited,"Gaya Market Gaya K.P. Road Gaya, Gaya, Bihar",Jay Private Limited Partners,"H.NO 51 GAYA MARET GAYA K.P. ROAD GAYA, GAYA, Bihar"
4,S1-453795736,S3-136583279,0.407674,Harris Twp Wildlife Direct Initiative,"607 Rosslyn Road, Harris Twp, PA",Harris Twp Wildlife Direct Initiative LP,"606 Rosslyn Rd, Harris Twp, Pennsylvania"
5,S1-479462718,S3-844680400,0.245957,Memorial Fellowship LLC,"109 Turkey Hollow Road, Campbell County, VA",The Memorial Fellowship LLC,
6,S1-593908410,S2-736814744,0.432852,Maid Auto Body,"825 Walker Avenue, Wenatchee, WA",Zephirixylo,"825 WALKER AVE, <NULL>, WENATCHEE, WA"
7,S1-634305339,S2-131572980,0.039538,Universal Exports Private Limited,"C-1, G-11, Ground Floor, Krishna Apra Plaza, Sector Alph...",यूनिवर्सल एक्सपोर्ट्स प्राइवेट लिमिटेड,"C-##1, LUCKNOW HQ REGION, उत्तर प्रदेश"
8,S1-718326235,S3-740299382,NaN,HHD Beverages Private Limited,"1/25, Hazuri Bhawan, Peepal Mandi Road, Agra, Uttar Pradesh",Ariapyra,"Agra, 1/25, UP, Agra"
9,S1-98315586,S2-315720146,0.355911,Delta Homecare,"1225 Osteen Street, Unit 1, Vidor, TX",Umbraectoorbi,"1225 OSTEEN ST, VIDOR, TX"


--- false_singleton: 1318 pairs (parent 1509)


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-15756653,S2-359256205,0.011222,Q F & B Biotherapeutics,"3418 Park Road, Spokane Valley, WA",Q F & B Biotherapeutics Ltd,"3685 PARK ROAD, SPOKANE VALLEY, WA"
1,S1-305224497,S2-544426290,NaN,Varois,"N3937 Schielke Road, Town Of Schley, WI",SOLZETA,"SCHIELKE RD, GLEASOON, WI"
2,S1-382106823,S3-193409076,0.024937,Twisted Grill LLC,"1067 Monroe Street, Chicago, IL",Twisted 6griatl LLC,
3,S1-433616795,S3-300908171,NaN,Cox Grand Paper Inc,"39 Sobro Avenue, Hempstead, NY",Cox Grand Pmep Inc,"#39 Sobro Avenue, Valley Stream, New York"
4,S1-60374765,S2-705571307,0.008954,Straight Edge Auto Body,"727 Millers Road, Des Plaines, IL",STRAIGHT EDGE AUTO BODY INC.,"997- MILLERS RD, DES PLAINES, IL"
5,S1-663224990,S2-244465048,0.393902,Coimbatore Marketing Group,"1/684 F, Sathi Main Road Kunnathur, Annur, Coimbatore, T...",Coimbatore Group-Center,"1677 F, SATHI MAIN ROAD KUNNATHUR, ANNUR, COIMBATORE, Ta..."
6,S1-74105174,S3-683300491,0.474284,Christensen Safe Drilling,"1587 Route 1, Perry, ME",Christensen Safe-Drilling,
7,S1-756901544,S3-181843208,0.313620,Best Mountain Commodities LLC,"991 Western Drive, Chanhassen, MN",Best Mountain,
8,S1-919213174,S3-411199990,0.086279,Poonam Jewelery Private Limited,"C-63, Surya Nagar, Ghaziabad, Uttar Pradesh",Poonam Jewe1ery Pridnnate Limited,"C-62, Surya Nagar, Ghaziabad, UP"
9,S1-982251971,S3-887774250,0.000285,Shiv Projects,"640, Block-O New Alipore, Kolkata, Kolkata, Howrah, West...",Shiv Projects,"40, Kolkota, পশ্চিমবঙ্গ"


--- singleton_merge: 314 pairs (parent 401)


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-101467335,S3-802180227,0.921432,Ag Road Co,"C/O- Prangya Paramita Mahanta, At- Brundaban Bihar, Madh...",Ag Road,
1,S1-181116332,S2-765632443,0.932349,Smyrna Animal Hospital Inc.,"110 Creek Court, Smyrna, TN",Smyrna Animal Hospital Inc,"TN, SMYRNA, 114 CREEK CT"
2,S1-30056058,S2-957804806,0.994960,Osprey Group,"231 Silvermine Avenue, Norwalk, CT",Osprey Group,"232 Silvermine Ave, NORWALK, CT"
3,S1-576813879,S2-871234060,0.660745,C 2 L Enbridge Inc,"309 Sunflower Dr, Fairfax, IA",C 2 L INC PARTNERS,
4,S1-602701852,S2-661502079,0.913087,Brightan Dogecoin LLC,"15 Eric Clauson Lane, Falmouth, MA",Brrightan Dogecoin Llc - 4697928860,"0015 ERIC CLAUSON LN, FALMOUHT CDP, MA"
5,S1-622878224,S2-208978463,0.988788,CZZ Solana Inc.,"MA, Franklin, 137 Union Street",CZZ Solana,
6,S1-718172744,S3-460238385,0.874551,Visoft Capital LLC,"4807 Cowslip Court, Oxon Hill, MD",Visoft Capital Partners,"4807 Cowslip Ct, Maryland, Oxon Hill"
7,S1-782452787,S2-206216664,0.549917,Smith Equity Partners LLC,"1116 7th Street, Havre, MT",SMITH EQUITY PARTNERS PARTNERS,"001121 SEVENTH ST, HAVRE, MT"
8,S1-963086391,S2-456895652,0.518203,All Clear Industrial,"1812 Beacon Street, Cincinnati, OH",All Clear,
9,S1-982362384,S3-883430624,0.620452,NX Neer Ltd,"13, Satyanarayan Temple Road Salkia, Howrah, West Bengal",NU Néer Ltd,"13, Satyanarayan Temple Road Salkia, Howrah, WB"


### 6.2 What the new features see, per outcome

Mean of every new feature over the val candidate pairs of four outcome classes: **TP** true pairs
predicted, **FN** true pairs among the candidates but not predicted, **FP** false merges
(predicted, not true, singleton merges included) and **TN** decoys, neither true nor predicted.
A feature that can still help has FP means unlike TP means (it flags the decoys the model still
merges) or FN means unlike TN means (it vouches for the true pairs still missed); a feature whose
FP and TP means coincide cannot remove the remaining false merges.

To stay cheap, the new features are rebuilt with `build_features(sample, s1n_val, pooln_val,
NEW_GROUPS)` on at most 300k pairs made of **whole S1 groups** (the `ctx_idf` ranks need every
candidate of a group), sorted by S1 id. Groups are drawn by id hash in two strata of half the
budget each, the groups holding an FP or FN row and the others, so the rare classes get enough
rows; each row weighs its stratum's group count over the groups drawn, which turns the means
back into estimates over all val candidate pairs. TN rows are capped at 200k. The pool
statistics are counted on the whole val pool, as in the scoring of §5.2, so the values equal
those the model saw.

In [16]:
DIAG_MAX_PAIRS = 300_000  # pairs featured for the diagnostic (whole S1 groups)
DIAG_MAX_TN = 200_000     # decoy rows kept for the TN means
DIAG_SEED = 40            # hash seed of the group draw: the same groups on every run
CLASSES = ("TP", "FN", "FP", "TN")


def sample_groups(pairs: pd.DataFrame, s1_ids: pd.Series, error_row: np.ndarray,
                  max_pairs: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Row mask of whole S1 groups drawn by id hash, and each row's weight.

    Two strata of groups get half of ``max_pairs`` each: the groups holding an error row and
    the others. Inside a stratum, groups are taken in the order of their id hash until its
    budget is spent, so no group is split, the sample stays within ``max_pairs`` and the draw
    depends on the ids only. A row weighs its stratum's group count over the groups taken, so
    weighted means undo the over-sampling of the error groups.
    """
    at = positions(pairs[C.S1_ID], s1_ids)  # S1 row of every pair
    if (at < 0).any():
        raise ValueError("pairs name S1 ids that are not in s1_ids")
    size = np.bincount(at, minlength=len(s1_ids))  # candidates per S1 group
    has_error = np.bincount(at, weights=error_row.astype(np.float64),
                            minlength=len(s1_ids)) > 0
    order = hash_unit(s1_ids, seed)
    keep, weight = np.zeros(len(s1_ids), dtype=bool), np.zeros(len(s1_ids))
    for stratum in (has_error, ~has_error & (size > 0)):
        members = np.flatnonzero(stratum)
        members = members[np.argsort(order[members], kind="stable")]
        taken = members[np.cumsum(size[members]) <= max_pairs // 2]  # a prefix: cumsum grows
        keep[taken] = True
        weight[taken] = len(members) / max(len(taken), 1)
    return keep[at], weight[at]


def class_means(X: pd.DataFrame, cls: np.ndarray, weight: np.ndarray) -> pd.DataFrame:
    """Weighted mean of every column per class of ``CLASSES`` (NaN values left out).

    ``cls`` holds the class code of each row (its position in ``CLASSES``; any other value
    leaves the row out); a column without a value in a class gets NaN there.
    """
    values = X.to_numpy(dtype=np.float64)
    out = {}
    for code, name in enumerate(CLASSES):
        x, w = values[cls == code], weight[cls == code][:, None]
        seen = ~np.isnan(x)
        with np.errstate(invalid="ignore", divide="ignore"):  # no value in the class: NaN
            out[f"{name} (n={len(x):,})"] = (np.where(seen, x * w, 0.0).sum(axis=0)
                                            / np.where(seen, w, 0.0).sum(axis=0))
    return pd.DataFrame(out, index=X.columns)


if not NEW_GROUPS:
    print("NEW_GROUPS is empty: nothing to diagnose")
else:
    is_pred = pair_in(val_pairs, val_matches)  # candidate pair predicted
    cls = np.select([is_true & is_pred, is_true & ~is_pred, ~is_true & is_pred], [0, 1, 2],
                    default=3).astype(np.int8)  # position in CLASSES
    in_sample, row_weight = sample_groups(val_pairs, val.s1[C.ENTITY_ID],
                                          (cls == 1) | (cls == 2), DIAG_MAX_PAIRS, DIAG_SEED)
    diag = (val_pairs[in_sample].assign(cls=cls[in_sample], weight=row_weight[in_sample])
            .sort_values(C.S1_ID, kind="stable"))  # whole groups, sorted by S1 id
    diag_cls = diag["cls"].to_numpy().copy()
    tn = np.flatnonzero(diag_cls == 3)
    if len(tn) > DIAG_MAX_TN:  # leave the surplus TN rows out of the means
        rng = np.random.default_rng(C.SEED)
        diag_cls[rng.choice(tn, len(tn) - DIAG_MAX_TN, replace=False)] = -1
    t0 = time.time()
    X_new = build_features(diag, s1n_val, pooln_val, NEW_GROUPS)  # stats: the whole val pool
    print(f"{len(diag):,} pairs of {diag[C.S1_ID].nunique():,} S1 groups featured in "
          f"{time.time() - t0:.0f} s; row weights "
          f"{np.unique(diag['weight'].to_numpy()).round(1).tolist()}")
    display(class_means(X_new, diag_cls, diag["weight"].to_numpy()).round(4))

299,959 pairs of 8,727 S1 groups featured in 17 s; row weights [8.6, 88.9]


,"TP (n=28,790)","FN (n=3,840)",FP (n=639),"TN (n=200,000)"
idf_name_cos,0.8352,0.6659,0.7968,0.3463
idf_name_top,0.5895,0.4276,0.5938,0.2418
idf_name_cover_l,0.8503,0.6797,0.8105,0.3503
idf_name_cover_r,0.8348,0.6563,0.7957,0.3631
idf_addr_cos,0.8764,0.6817,0.7535,0.2849
idf_addr_top,0.7215,0.6470,0.6794,0.4172
idf_addr_cover_l,0.8720,0.6792,0.7535,0.3114
idf_addr_cover_r,0.9125,0.7734,0.8244,0.3484
freq_name_l,13.7799,24.4495,7.5227,17.3218
freq_name_r,11.3472,16.2066,5.2410,17.5626


### 6.3 Transitions from the parent

When the parent's `val_matches.parquet` exists: the pairs this version fixed (a parent false
merge no longer predicted, a parent miss now predicted) and broke (a new false merge, a parent
true pair lost), the entities whose F0.5 rose or fell, and a sample of the new false merges, the
costliest change under F0.5. Skipped, with a message, when the file is missing.

In [17]:
if parent_matches is None:
    print(f"transitions skipped: {parent_matches_path} does not exist on this machine")
else:
    this_true = pair_in(val_matches, val.pairs)
    this_in_parent = pair_in(val_matches, parent_matches)
    parent_true = pair_in(parent_matches, val.pairs)
    parent_in_this = pair_in(parent_matches, val_matches)
    display(pd.Series({
        "fixed: parent false merge dropped": int((~parent_true & ~parent_in_this).sum()),
        "fixed: parent miss now predicted": int((this_true & ~this_in_parent).sum()),
        "broken: new false merge": int((~this_true & ~this_in_parent).sum()),
        "broken: parent true pair lost": int((parent_true & ~parent_in_this).sum()),
    }, name="pairs").to_frame())
    f_this = entity_f05_from_counts(*entity_counts(val_matches, val))
    f_parent = entity_f05_from_counts(*entity_counts(parent_matches, val))
    delta = f_this - f_parent
    print(f"entities: {int((delta > 0).sum()):,} improved, {int((delta < 0).sum()):,} worsened; "
          f"macro F0.5 {delta.mean():+.5f}")
    new_fp = val_matches[~this_true & ~this_in_parent]  # predicted now, false, not by parent
    display(pd.concat([error_samples(new_fp, val, kind, n=5, scored=val_scored)
                       for kind in ("false_merge", "singleton_merge")], ignore_index=True))

,pairs
fixed: parent false merge dropped,3520
fixed: parent miss now predicted,13314
broken: new false merge,1769
broken: parent true pair lost,5421


entities: 15,675 improved, 6,774 worsened; macro F0.5 +0.00261


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-104216715,S2-928685796,0.608247,Toss Holdings,"60, 2Nd Main Road, Thirumalai Nagar Annex, Saidapet, Kan...",Toss Systems,"67, 2ND MAIN ROAD, THIRUMALAI NAGAR ANNE, SAIDAPET, Tami..."
1,S1-514791271,S3-916567280,0.463590,Trading Competition Vision Limited,"521A Lane No 23Western Avenur Sainik Farm, New Delhi, So...",Trading Competition,
2,S1-557029797,S3-570082564,0.753184,Hein & Ohara Corp,"48 Auburn Road, Peru, ME",Hein & Ohara Corp,
3,S1-625512706,S3-793416377,0.838940,Kent Future Inc,"3227 67th Terrace, Kansas City, KS",Kent Future,
4,S1-915651212,S2-941036462,0.541439,Shyam Tech Private Limited,"6-20-83, 9/1, Arundelpet, Guntur, Andhra Pradesh",SHYAM TECH PRIVATE LIMITED Enterprises,
5,S1-576813879,S2-871234060,0.660745,C 2 L Enbridge Inc,"309 Sunflower Dr, Fairfax, IA",C 2 L INC PARTNERS,
6,S1-622878224,S2-208978463,0.988788,CZZ Solana Inc.,"MA, Franklin, 137 Union Street",CZZ Solana,
7,S1-690763475,S3-738208639,0.752688,Alpha Maa Systems Private Limited,"A1010 10Th Floor 9 Square, Nana Mava Main Road, Rajkot, ...",Alpha Maa Systems LLP,"Surat, ગુજરાત, A1010 10Th Floor 10 Square"
8,S1-716971268,S3-691830773,0.818280,"Traa, LLC","3400 Bowman Road, Unit Apt 316, Little Rock, AR",Traa LLC,"738 36th Street, Little Rock, Arkansas"
9,S1-963086391,S2-456895652,0.518203,All Clear Industrial,"1812 Beacon Street, Cincinnati, OH",All Clear,


## 7. Log the result

Records `metrics.json` and this version's row in `experiments/experiments.csv`, stamped with the
git commit of `src/` (it must not end in `-dirty`: commit the feature code before running). The
record holds the 13 §2.2 keys as in v001, plus `max_slice_drop`, `importance_top20` and
`new_feature_importance`. The decision is the 13 §3 rule, computed rather than typed:

* **KEEP**: `f_beta` > parent + 0.002, `max_slice_drop` ≤ 0.01 (or unavailable) and
  `harder_f_beta` ≥ the parent's;
* **DROP**: `f_beta` ≤ parent, or a slice down by more than 0.01;
* **INVESTIGATE**: anything else (above the parent by less than the margin, or harder-val down).
  Run time and RSS (§5.5) are judged by hand in §8.

In [18]:
MARGIN, SLICE_TOL = 0.002, 0.01  # 13 §3: KEEP margin over the parent, largest slice drop
f_beta, parent_f = metrics["f_beta"], parent_metrics["f_beta"]
harder = harder_metrics["f_beta"]
parent_harder = float(parent_metrics.get("harder_f_beta", np.nan))  # NaN: no harder check
slices_ok = np.isnan(max_slice_drop) or max_slice_drop <= SLICE_TOL
harder_ok = np.isnan(parent_harder) or harder >= parent_harder
if f_beta > parent_f + MARGIN and slices_ok and harder_ok:
    DECISION = "KEEP"
elif f_beta <= parent_f or not slices_ok:
    DECISION = "DROP"
else:
    DECISION = "INVESTIGATE"
print(f"f_beta {f_beta:.4f} vs parent {parent_f:.4f} ({f_beta - parent_f:+.4f}; KEEP needs "
      f"> +{MARGIN}); max_slice_drop {max_slice_drop:.4f} (<= {SLICE_TOL}); harder "
      f"{harder:.4f} vs {parent_harder:.4f} -> {DECISION}")

record = {
    "hypothesis": HYPOTHESIS,
    "blocking_config": asdict(cfg.blocking), "feature_groups": list(cfg.feature_groups),
    "model_params": asdict(cfg.model), "rule": asdict(fitted.rule),
    **{k: metrics[k] for k in ("f_beta", "f_beta_singletons", "f_beta_matched",
                               "pair_precision", "pair_recall")},
    **{k: metrics[k] for k in ("cand_recall", "entity_recall", "ceiling_f_beta",
                               "cands_mean", "cands_p95")},
    "harder_f_beta": harder_metrics["f_beta"],
    "tune_f_beta": float(fitted.tune_table["f_beta"].max()),
    "n_fp": counts["false_merge"] + counts["singleton_merge"], "n_fn": counts["missed"],
    "n_false_singleton": counts["false_singleton"], "errors": counts,
    "token_map_size": fitted.info["token_map_size"],
    "best_iteration": fitted.matcher.best_iteration_,
    **timings, "peak_rss_gb": peak_rss_gb(), "decision": DECISION,
    "max_slice_drop": None if np.isnan(max_slice_drop) else max_slice_drop,
    "importance_top20": {f: float(g) for f, g in importance["gain"].head(20).items()},
    "new_feature_importance": {f: {"group": r["group"], "rank": int(r["rank"]),
                                   "gain": float(r["gain"])}
                               for f, r in new_importance.iterrows()},
}
slice_note = "n/a" if np.isnan(max_slice_drop) else f"{max_slice_drop:+.4f}"
row = log_result(
    EXP_DIR, change=CHANGE, group=GROUP, local_f05=metrics["f_beta"],
    cand_recall=metrics["cand_recall"],
    notes=(f"cands {metrics['cands_mean']:.1f}/S1; singleton F0.5 "
           f"{metrics['f_beta_singletons']:.4f}; harder {harder_metrics['f_beta']:.4f}; "
           f"{f_beta - parent_f:+.4f} vs {PARENT.split('_')[0]}; max slice drop {slice_note}"),
    metrics=record, owner=OWNER, parent=PARENT.split("_")[0], decision=DECISION)
print(row)

f_beta 0.9870 vs parent 0.9844 (+0.0026; KEEP needs > +0.002); max_slice_drop -0.0019 (<= 0.01); harder 0.9864 vs 0.9838 -> KEEP
{'version': 'v040', 'date': '2026-09-26', 'group': 'C2', 'change': 'v001 + idf, frequency, ctx_idf, address_extra feature groups', 'local_f05': '0.9870', 'cand_recall': '0.9906', 'public_f05': '', 'commit': '6e7b015', 'notes': 'cands 33.0/S1; singleton F0.5 0.9875; harder 0.9864; +0.0026 vs v001; max slice drop -0.0019', 'owner': 'M3', 'parent': 'v001', 'decision': 'KEEP'}


## 8. Conclusion

* **Result vs parent (§5.5):** local macro F0.5 0.98436 → **0.98698 (+0.00261)**; singletons
  0.98400 → 0.98752 (+0.00352), matched 0.98439 → 0.98695 (+0.00256); pair precision 0.99521 →
  0.99641 and pair recall 0.96373 → 0.96889, both up; harder-val 0.98383 → 0.98643 (+0.00260);
  tune 0.98465 → 0.98717. Candidates unchanged (`cand_recall` 0.99057, ceiling 0.99707).
* **Decision (§7): KEEP.** Margin +0.00261 > 0.002; no slice fell (`max_slice_drop` −0.0019:
  the smallest change is a gain, `n_matches` 5+); harder-val above the parent. The parent's
  slices come from a same-machine re-run of v001 that reproduced its logged scores exactly
  (0.98436, harder 0.98383, same rule and best iteration).
* **Slices (§5.6):** the gain is largest where predicted: `addr_empty = yes` +0.0068 (0.9583 →
  0.9651), `n_matches = 1` +0.0054, source S2 +0.0041, singletons +0.0035; `ambiguous = yes`
  +0.0029, slightly above the overall +0.0026. No slice lost.
* **New features (§5.1, §6.2):** the 23 new columns take 15.6 % of the gain: `idf` 6.5 %,
  `address_extra` 5.6 %, `ctx_idf` 1.8 %, `frequency` 1.7 %. In the top 20: `num_contain_l`
  (#7), `idf_addr_cos` (#8), `ad_contain_r` (#12), `ctx_rank_idf_addr` (#14),
  `idf_addr_cover_r` (#15), `num_contain_r` (#17). Number containment separates false merges
  from true pairs best (`num_contain_l` FP 0.62, TP 0.89, TN 0.08), then the address idf rank
  (`ctx_rank_idf_addr` TP 3.3, FP 15.0). Zero gain: `addr_empty_l` (no empty S1 address in
  val) and `postcode_prefix_eq` (postcodes are almost always empty: 438 entities have one),
  besides v001's `sim_addr_char`, `sorted_eq`, `addr_empty_r`.
* **Errors (§6.1, §6.3):** false merges 6,685 → 5,021, misses 53,904 → 46,202, false
  singletons 1,509 → 1,318, singleton merges 401 → 314. Pairs: 3,520 false merges and 13,314
  misses fixed, 1,769 new false merges and 5,421 true pairs lost; 15,675 entities improved,
  6,774 worsened. The new false merges are mostly pool records with an empty address and a
  shortened name (`Toss Systems`, `Kent Future`, `All Clear`): name-only evidence stays the
  weak spot.
* **Cost (§5.5):** val scoring 814 s against 459 s for the same-machine v001 re-run (1.8×; the
  logged 270 s ran without memory pressure), fit 496 s against 439 s, peak RSS 5.28 GB. About
  1.8× the scoring time at test scale: acceptable.
* **Pool-size risk (§1):** `frequency` carries only 1.7 % of the gain. On val the model reads it
  as decoy evidence (`freq_name_l` 24.4 on misses, 13.8 on true pairs), but its counts come
  from a 6.2M-record fit pool and are applied to 2.1M (val) and per-country test pools.
* **Next experiment:** v041 = v001 + `idf`, `ctx_idf`, `address_extra` (no `frequency`). If it
  stays within ~0.0005 of v040, prefer it for test, since it has no raw pool counts. Later:
  drop `addr_empty_l` and `postcode_prefix_eq` once they show zero gain in three versions.


## 9. Test inference (shortlisted versions only)

Runs only with `RUN_TEST = True` (parameter cell), set when M1 picks this version for an upload
(13 §5); otherwise the cells print a message and write nothing. Same pipeline on the test split:
normalise (cached), block per country (France included), score (the new groups' pool statistics
counted on each country's test pool), decide with the frozen rule, and write both files with
`submission.write_pairs` from the exact pairs frame that was scored. Then the sanity checks of
11 §10: one row per test S1 in both files, every country present with candidates and matches,
match rates and candidates per S1 close to val.

In [19]:
def per_country(s1n: pd.DataFrame, matches: pd.DataFrame,
                n_cands_by_s1: pd.Series) -> pd.DataFrame:
    """Match rate, matches and candidates per S1, by country (11 §10 sanity table)."""
    country = s1n.set_index(C.ENTITY_ID)[C.COUNTRY]
    n_s1 = s1n.groupby(C.COUNTRY).size()
    by = matches[C.S1_ID].map(country)
    return pd.DataFrame({
        "s1": n_s1,
        "cands_per_s1": n_cands_by_s1.groupby(n_cands_by_s1.index.map(country)).sum() / n_s1,
        "matched_share": matches.groupby(by)[C.S1_ID].nunique() / n_s1,
        "matches_per_s1": matches.groupby(by).size() / n_s1,
    })


if RUN_TEST:
    t0 = time.time()
    match_path, cand_path, s1n_test, test_matches, test_summary = run_test(cfg, fitted)
    print(f"run_test {time.time() - t0:.0f} s -> {match_path}, {cand_path}")
    test_table = per_country(s1n_test, test_matches, test_summary["n_cands"])
    val_table = per_country(s1n_val, val_matches, val_pairs.groupby(C.S1_ID).size())
    display(pd.concat({"test": test_table, "val": val_table}))
    country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
    display(pd.crosstab(test_summary.index.map(country_of),
                        pd.cut(test_summary["p_max"], [0, .1, .3, .5, .7, .9, 1.0]),
                        normalize="index").round(3))
else:
    print("RUN_TEST is False: no test inference (only shortlisted versions write output/)")

RUN_TEST is False: no test inference (only shortlisted versions write output/)


Both validators on the files that would be uploaded: ours (`submission.validate` with id
existence checks) and the organisers' stdlib-only `validate_submission.py`. After a PASS: `make
validate`, the upload, the `LEADERBOARD.md` entry and `make public V=v040 SCORE=0.xxxx`.

In [20]:
if RUN_TEST:
    out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                          str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
    print(out.stdout[-2000:], out.stderr[-2000:])
    out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                          "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                         capture_output=True, text=True)
    print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

notebook total 2717 s, peak RSS 5.28 GB
